# 00 publications (g1)

Every list we have seen, one row per distinct file. What you should see: 8 rows, one per source; `as_of_ok` and `hash_ok` true on every row; EirGrid as of 2026-08-05, ESB as of 2026-07-01.

In [ ]:
import subprocess
from datetime import datetime
from pathlib import Path

from iegrid import store
from iegrid.checks import check_publications, table

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / store.DEFAULT_DB
sha = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(DB)
last = con.execute("select max(ingested_at) from publication").fetchone()[0]
print(f"store {DB.name}  last ingest {last:%Y-%m-%d %H:%M} UTC  git {sha}")

In [ ]:
print(table(con, """
    select source_id, as_of, published_at, published_at_basis, vintage_quality,
           round(size_bytes / 1024) as kb, left(sha256, 12) as sha
    from publication order by source_id, as_of
"""))

## done-when: as-of in the PDF matches the date in the file name, raw file hash matches

In [ ]:
import os
os.chdir(ROOT)  # raw_path is relative to the repo root
checks = check_publications(con)
for c in checks:
    print(f"{c.source_id:34} pdf {c.as_of}  name {c.as_of_from_name}  as_of_ok {c.as_of_ok}  hash_ok {c.hash_ok}")
assert checks, "no publications stored"
assert all(c.as_of_ok for c in checks), "as-of in the PDF disagrees with the file name"
assert all(c.hash_ok for c in checks), "a raw file is missing or changed"
print(f"\n{len(checks)} publications, all checks pass")

## lag: how long after the as-of date each list went up

In [ ]:
print(table(con, """
    select source_id, as_of, published_at::date as published,
           date_diff('day', as_of, published_at::date) as days_after_as_of
    from publication order by source_id
"""))